# Error Handling — Full-Time Edition

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- explain what an exception is;
- recognize common exceptions such as `ValueError`, `TypeError`, `KeyError`, and `IndexError`;
- use `try` / `except` to handle expected errors;
- catch a **specific** exception type;
- access the original exception message with `as`;
- distinguish between a Python error and a value that violates a business rule;
- use `raise` to enforce a rule;
- combine `while` with `try/except` so a program keeps asking until valid input is entered;
- validate whether a user choice belongs to a set of allowed options.

> **Main idea:** error handling is not about hiding errors.  
> It is about deciding what your program should do when something goes wrong.


## 1. What can go wrong?

So far, many of our programs assume that users enter exactly what we expect.

Imagine an event-booking app asking for the number of attendees:


In [ ]:
attendees = int(input("How many attendees are coming? "))


If the user types:

```text
twenty
```

Python cannot convert that string to an integer and raises a `ValueError`.

Without error handling, the normal execution of the program stops.


### Common exceptions

You do **not** need to memorize every Python exception.

For now, recognize these common ones:

- `ValueError` → the value cannot be used as expected;
- `TypeError` → an operation receives an incompatible type;
- `KeyError` → a dictionary key does not exist;
- `IndexError` → a sequence index does not exist.

For this lesson, **`ValueError` is the most important one**, because user input often needs conversion.


### ⚡ Quick check — predict the exception

Predict the exception before running the cell:


In [ ]:
age = int("twenty")


## 2. `try` / `except`

Put code that may fail inside `try`.

If the expected exception occurs, Python jumps to the matching `except` block.


In [ ]:
try:
    attendees = int(input("How many attendees are coming? "))
    print(f"Registering {attendees} attendees.")

except ValueError:
    print("Please enter the number using digits.")


The core pattern is:

```python
try:
    # risky operation

except ExpectedError:
    # what to do if that error occurs
```


### ⚡ Quick check — complete the handler


In [ ]:
try:
    year = int(input("Enter the event year: "))
    print(f"Searching events in {year}...")

except ________:
    print("Please enter a valid year.")


## 3. Catch the error you expect

Prefer:

```python
except ValueError:
```

to a bare:

```python
except:
```

when you know which error is expected.

Catching a specific exception makes the intention clearer and avoids hiding unrelated bugs.


### Access the original message with `as`

The exception itself can be stored in a variable:


In [ ]:
try:
    row = int("front")

except ValueError as error:
    print(f"Could not convert the row: {error}")


The name `error` is just a variable name. You could choose another name:

```python
except ValueError as exc:
```

The important part is that the variable contains the exception object and its message.


### ⚡ Quick check — reuse the error message


In [ ]:
try:
    guests = int("many")

except ValueError as ________:
    print(f"Invalid number of guests: {________}")


## 4. Python errors vs business-rule errors

Not every invalid value causes Python itself to fail.

For example:


In [ ]:
tickets = int("-3")
print(tickets)


Python is perfectly able to convert `"-3"` into the integer `-3`.

But in a ticket-booking application, a negative number of tickets makes no sense.

This is a **business-rule error**, not a conversion error.

We can detect the rule ourselves and deliberately raise an exception:


In [ ]:
tickets = -3

if tickets < 0:
    raise ValueError("The number of tickets cannot be negative.")


This distinction is important:

```text
int("three")
    ↓
Python detects the problem
    ↓
ValueError


int("-3")
    ↓
Python accepts it
    ↓
our program rejects it
    ↓
raise ValueError(...)
```


### ⚡ Quick check — enforce a rule

A rental service accepts durations up to 30 days.

Complete the condition:


In [ ]:
days = 45

if ________:
    raise ValueError("Rental duration cannot exceed 30 days.")


## 5. The key pattern: `while` + `try/except`

Handling an error once is useful.

Interactive programs often need something more:

> **Ask again until the user enters a valid value.**

The pattern is:

1. start a loop;
2. ask for input inside `try`;
3. convert and validate the value;
4. leave the loop only when the value is valid;
5. if an expected error occurs, explain the problem and repeat.


In [ ]:
valid_input = False

while not valid_input:
    try:
        attendees = int(input("How many attendees are coming? "))

        if attendees < 0:
            raise ValueError("Number of attendees cannot be negative.")

        valid_input = True

    except ValueError as error:
        print(f"Invalid input: {error}")

print(f"Registering {attendees} attendees.")


### Why does this work?

```text
valid_input = False
        ↓
while not valid_input
        ↓
      try
        ↓
conversion / validation
   ┌────┴─────┐
 valid      invalid
   │            │
   ▼            ▼
True         except
   │            │
exit loop    print error
                │
                └── repeat
```

The loop ends only after **all checks pass**.


### ⚡ Quick check — complete the retry loop

A coworking space only accepts bookings of at least 1 hour.


In [ ]:
valid_duration = False

while not valid_duration:
    try:
        hours = int(input("How many hours do you want to book? "))

        if ________:
            raise ValueError("Booking duration must be at least 1 hour.")

        ________ = True

    except ________ as error:
        print(f"Invalid duration: {error}")


## 6. One `try` block can have several specific `except` blocks

Sometimes one risky block can fail in more than one expected way.

In that case, we can use one `except` per error type:


In [ ]:
settings = {"capacity": 40}

try:
    key = input("Which setting do you want to read? ")
    value = settings[key]
    numeric_value = int(value)

except KeyError:
    print("That setting does not exist.")

except ValueError:
    print("The selected setting cannot be converted to an integer.")


This is enough for now:

> **one `except` per expected error type**

We do not need more advanced `try` syntax for today's Lab.


## 7. Validate a choice

Sometimes the input has the correct type but is not one of the allowed values.

We can combine membership checking with `raise`:


In [ ]:
valid_cities = ["Madrid", "Barcelona", "Valencia"]

valid_city = False

while not valid_city:
    try:
        city = input("Choose a destination: ")

        if city not in valid_cities:
            raise ValueError("Destination is not available.")

        valid_city = True

    except ValueError as error:
        print(error)

print(f"Destination selected: {city}")


### ⚡ Quick check — validate membership


In [ ]:
meal_options = ["vegetarian", "vegan", "standard"]

valid_meal = False

while not valid_meal:
    try:
        meal = input("Choose your meal: ").lower()

        if ________:
            raise ValueError("Meal option not available.")

        valid_meal = True

    except ValueError as error:
        print(error)


## 8. Put the pattern inside a function

Error handling often lives inside a function whose job is to obtain or validate data.

Remember the rule from our Functions lesson:

> If a function needs external data → **parameter**.  
> If useful data is created inside the function and needed outside → **return**.


In [ ]:
def ask_for_capacity(room_name):
    valid_capacity = False

    while not valid_capacity:
        try:
            capacity = int(
                input(f"Enter the capacity for {room_name}: ")
            )

            if capacity < 0:
                raise ValueError("Capacity cannot be negative.")

            valid_capacity = True

        except ValueError as error:
            print(f"Invalid capacity: {error}")

    return capacity


main_room_capacity = ask_for_capacity("Main Room")
print(main_room_capacity)


Notice the responsibilities:

```text
room_name
    ↓ parameter
function
    ↓
ask + validate
    ↓
capacity
    ↓ return
```

The error-handling logic stays inside the function, while the valid result is returned to the rest of the program.


## 🧩 Mini Challenge — robust event registration

Create a function:

```python
get_valid_age()
```

The function should keep asking for an attendee's age until:

- the input can be converted to an integer;
- the age is between `0` and `120`.

Use:

- `while`;
- `try`;
- `except ValueError`;
- `raise ValueError(...)`;
- `return`.

Do not build a complete application. Focus only on making one input robust.


In [ ]:
# Your solution here


# 🚀 Lab Readiness Check

Before starting the Lab, make sure you can answer **yes** to these questions:

- [ ] Can I recognize a `ValueError` caused by invalid input?
- [ ] Can I put risky code inside `try`?
- [ ] Can I catch a specific error with `except ValueError`?
- [ ] Can I use `as error` to access the exception message?
- [ ] Can I distinguish a Python conversion error from a business-rule error?
- [ ] Can I use `raise ValueError(...)` to enforce a rule?
- [ ] Can I combine `while` with `try/except` to keep asking until input is valid?
- [ ] Can I validate that a string belongs to a collection of allowed values?
- [ ] Can I use one `except` block per expected error type?
- [ ] Can I return the valid result from a function?

If yes, you have the tools needed to make an existing interactive program more robust.


---

# 📚 Optional — self-learning / reference

The following features are useful, but they are **not required for today's Lab**.

## `else`

A `try` block may include an `else` section that runs only if no exception occurs.

```python
try:
    value = int("42")
except ValueError:
    print("Invalid value")
else:
    print("Conversion succeeded")
```

## `finally`

A `finally` block runs whether an exception occurs or not.

```python
try:
    value = int("42")
except ValueError:
    print("Invalid value")
finally:
    print("Validation attempt finished")
```

This is useful when some cleanup action must always happen.

## Other topics for later

- `assert`;
- custom exception classes;
- the Python exception hierarchy;
- re-raising exceptions;
- more advanced exception-handling patterns.

These are useful Python features, but they are not needed for today's Lab.
